# Prompt Engineering 101 - Part V. HOMEWORK
## Knowledge & Synthesis (RAG)

In [ ]:
# @title 🛠️ Step 1: Laboratory Setup (Gemini + FAISS + Vision)
# We are installing:
# 1. Google GenAI (The Model)
# 2. FAISS (The Vector Database)
# 3. Sentence-Transformers (The Embedding Tool)

# 1. Install the dependencies
!pip install -q -U google-genai faiss-cpu sentence-transformers


import base64
import requests

import faiss
from sentence_transformers import SentenceTransformer

from google.colab import userdata
from google import genai

from IPython.display import display, Markdown, Image


# 2. Configure the API Key
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')


# 3. Create a Robust Wrapper Class
class GeminiModel:
    def __init__(self, api_key, preferred_model='gemini-2.5-flash'):
        self.client = genai.Client(api_key=api_key)
        
        # Priority list of models and their availability status
        # True = Available, False = Exhausted (429)
        self.models = {
            'gemini-2.5-flash': True,
            'gemini-2.5-flash-lite': True,
            'gemini-3-flash-preview': True,
        }
        
        # Validation: Ensure the user's choice is valid
        if preferred_model not in self.models:
            raise ValueError(f"Invalid model '{preferred_model}'. "
                             f"Valid options: {list(self.models.keys())}")
            
        self.current_model = preferred_model

    def _get_next_available_model(self):
        """Sets the first model from the non-exhausted models as the currently selected model. 
        Raises error if no available model left."""
        for model_name, is_available in self.models.items():
            if is_available:
                print(f"🔄 Switching to fallback: {model_name}")
                self.current_model = model_name
                return 
    
        raise RuntimeError("❌ All available models are currently exhausted. "
                           "Please wait for quotas to reset.")

    def generate_content(self, contents, config=None):
        """
        Recursively attempts to generate content.
        If a model fails with a quota error, it marks it as unavailable 
        and retries with the next available model.
        """
        try:
            # Attempt generation
            response = self.client.models.generate_content(
                model=self.current_model,
                contents=contents,
                config=config
            )
            return response

        except Exception as e:
            # Check for Rate Limit / Quota errors
            if "429" in str(e) or "ResourceExhausted" in str(e):
                print(f"⚠️ Quota exceeded for {self.current_model}. Marking as unavailable...")
                
                # Update State: Mark current as failed
                self.models[self.current_model] = False
                
                # Switch to next available
                self._get_next_available_model()
            
                # Recursive Step: Try again with the new model
                return self.generate_content(contents, config)
            
            # If it's a logic error (e.g. invalid prompt), raise immediately
            raise e


# 4. Initialize
try:
    model = GeminiModel(GEMINI_API_KEY, preferred_model='gemini-2.5-flash')
    print(f"✅ Connection Established. Using {model.current_model}.")
except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
# @title 🖼️ Topic 2: Vision Helper Function
# We need a tool to send images to the AI.
# This function handles the messy parts (Download -> Base64 -> API).

def analyze_image_from_url(image_url, prompt):
    try:
        # 1. Download Image
        response = requests.get(image_url)
        response.raise_for_status()

        # 2. Prepare for API (Gemini expects specific dict format)
        image_part = {
            "mime_type": "image/jpeg",
            "data": base64.b64encode(response.content).decode('utf-8')
        }

        # 3. Send to Model
        print("Analyzing image...")
        model_response = model.generate_content([prompt, image_part])
        return model_response.text

    except Exception as e:
        return f"Error: {e}"

print("✅ Vision Tool Loaded.")

# 🏠 Homework: The Insurance Claim Bot

### The Scenario
You work at an Insurance Company.
A customer has submitted a claim for a car accident.
They uploaded a photo of the damage.

### The Task
Write a script that:
1.  **Analyzes an Image** of a car crash (use the URL below).
    * *URL:* `https://upload.wikimedia.org/wikipedia/commons/a/a6/Bumper_dent.JPG`
    * *Prompt:* "Identify the specific part of the car that is damaged."
2.  **Searches the Web** for the replacement cost.
    * *Search:* "Replacement cost for [DAMAGED_PART] 2022 Toyota"
3.  **Generates a Report** combining the visual damage and the estimated cost.

### Submission
Submit the Python code and the final text report.

In [ ]:
# @title (Hidden) Answer Key

from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

pitch_deck = "Our product, the SmartToast AI, is the FIRST and ONLY AI toaster in the world."

investigator_prompt = f"""
CLAIM FROM PITCH DECK: "{pitch_deck}"

TASK:
1. Search Google for "AI powered toaster competitors".
2. Compare the search results to the claim.
3. Is the claim "First and Only" true? Write a Risk Assessment.
"""

# Enable Search
search_tool = Tool(google_search=GoogleSearch())
config = GenerateContentConfig(tools=[search_tool])

print(model.generate_content(investigator_prompt, config=config).text)